# PathFinder Main Notebook

This notebook mirrors `main.py` with a notebook-local training loop that uses `tqdm.auto` so you can watch progress live.


In [1]:
from pathlib import Path

import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.tensorboard import SummaryWriter
from tqdm.auto import tqdm

from evaluate import evaluate, evaluate_top_k
from graph_builder import build_graph
from model import Model
from train import split_data, traintestval_loader, ts_loader

d:\Personal\code\path_model\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
d:\Personal\code\path_model\.venv\Lib\site-packages\torch_geometric\__init__.py:4: UserWarning: An issue occurred while importing 'torch-sparse'. Disabling its usage. Stacktrace: [WinError 127] The specified procedure could not be found
  import torch_geometric.typing


In [2]:
config = {
    "data_dir": "D:/Personal/code/path_model/PathFinder/required_data",
    "output_dir": "output",
    "experiment_name": "hardcoded_shortlist_job_notebook",
    "predict_edge": ("candidature", "has_application", "job"),
    "use_candidature_node": True,
    "use_temporal": True,
    "use_ts_loader": True,
    "ts_nodes_all": True,
    "remove_feat": False,
    "freeze": False,
    "save_model": True,
    "seed": 42,
    "hidden_channels": 64,
    "lr": 1e-4,
    "wd": 1e-5,
    "batch_norm": "layer_norm",
    "linear_unit": "relu",
    "num_layers": 3,
    "conv_operator": "gat",
    "strategy": "uniform",
    "num_neigh": [20, 10],
    "batch_size": 128,
    "eval_batch_size": 3 * 128,
    "train_metric": 0,
    "max_epoch": 30,
    "max_val_decrease": 20,
    "model_list_abl": [1, 1, 1, 1, 1, 1, 1, 1, 1],
    "error_analysis": False,
}

torch.manual_seed(config["seed"])
output_dir = Path(config["output_dir"])
output_dir.mkdir(parents=True, exist_ok=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

config


{'data_dir': 'D:/Personal/code/path_model/PathFinder/required_data',
 'output_dir': 'output',
 'experiment_name': 'hardcoded_shortlist_job_notebook',
 'predict_edge': ('candidature', 'has_application', 'job'),
 'use_candidature_node': True,
 'use_temporal': True,
 'use_ts_loader': True,
 'ts_nodes_all': True,
 'remove_feat': False,
 'freeze': False,
 'save_model': True,
 'seed': 42,
 'hidden_channels': 64,
 'lr': 0.0001,
 'wd': 1e-05,
 'batch_norm': 'layer_norm',
 'linear_unit': 'relu',
 'num_layers': 3,
 'conv_operator': 'gat',
 'strategy': 'uniform',
 'num_neigh': [20, 10],
 'batch_size': 128,
 'eval_batch_size': 384,
 'train_metric': 0,
 'max_epoch': 30,
 'max_val_decrease': 20,
 'model_list_abl': [1, 1, 1, 1, 1, 1, 1, 1, 1],
 'error_analysis': False}

In [3]:
def train_notebook(
    model,
    train_loader,
    val_loader,
    test_loader,
    all_data,
    name_experiment,
    strategy,
    output_dir,
    wd=0.00001,
    predict_edge=("candidature", "has_application", "job"),
    num_neigh=None,
    use_ts_loader=True,
    save_model_bool=True    ,
    train_metric=0,
    max_epoch=30,
    max_val_decrease=20,
    lr=0.0001,
    hyperparameters=None,
    error_analysis=False,
):
    if num_neigh is None:
        num_neigh = [20, 10]
    if hyperparameters is None:
        hyperparameters = {}

    writer = SummaryWriter(output_dir + f"/runs/{name_experiment}")
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=wd)

    model.train()
    best_metric_early_stop = 100 if train_metric == 0 else -1
    counter = 0

    auc, precision, recall, acc, f1, val_loss = evaluate(model, val_loader, predict_edge=predict_edge)
    writer.add_scalar("Loss/val", val_loss, 0)
    writer.add_scalar("Metrics/auc", auc, 0)
    writer.add_scalar("Metrics/precision", precision, 0)
    writer.add_scalar("Metrics/recall", recall, 0)
    writer.add_scalar("Metrics/accuracy", acc, 0)
    writer.add_scalar("Metrics/F1", f1, 0)

    epoch_bar = tqdm(range(max_epoch), desc="Epochs")
    for epoch in epoch_bar:
        if counter >= max_val_decrease:
            break

        total_loss = 0
        total_examples = 0
        all_batch_size = []

        batch_bar = tqdm(train_loader, leave=False, desc=f"Epoch {epoch:03d}")
        for sampled_data in batch_bar:
            optimizer.zero_grad()
            sampled_data = sampled_data.to(device)
            pred = model(sampled_data)
            ground_truth = sampled_data[predict_edge[0], predict_edge[1], predict_edge[2]].edge_label

            all_batch_size.append(len(ground_truth))

            loss = F.binary_cross_entropy_with_logits(pred.float(), ground_truth.float())
            loss.backward()
            optimizer.step()

            total_loss += float(loss) * pred.numel()
            total_examples += pred.numel()
            batch_bar.set_postfix(loss=float(loss))

        train_loss = total_loss / total_examples
        auc, precision, recall, acc, f1, val_loss = evaluate(model, val_loader, predict_edge=predict_edge)

        writer.add_scalar("Loss/train", train_loss, epoch)
        writer.add_scalar("Loss/val", val_loss, epoch)
        writer.add_scalar("Metrics/auc", auc, epoch)
        writer.add_scalar("Metrics/precision", precision, epoch)
        writer.add_scalar("Metrics/recall", recall, epoch)
        writer.add_scalar("Metrics/accuracy", acc, epoch)
        writer.add_scalar("Metrics/F1", f1, epoch)

        if train_metric == 0:
            metric_early_stop = val_loss
        elif train_metric == 1:
            metric_early_stop = f1
        elif train_metric == 2:
            metric_early_stop = auc
        else:
            metric_early_stop = None

        improved = False
        if train_metric == 0:
            if val_loss < best_metric_early_stop:
                best_metric_early_stop = metric_early_stop
                improved = True
        else:
            if metric_early_stop > best_metric_early_stop:
                best_metric_early_stop = metric_early_stop
                improved = True

        if improved:
            counter = 0
            if save_model_bool:
                Path(output_dir + f"/models_temp/{name_experiment}").mkdir(parents=True, exist_ok=True)
                torch.save(model, output_dir + f"/models_temp/{name_experiment}/model.pt")
        else:
            counter += 1

        epoch_bar.set_postfix(
            train_loss=f"{train_loss:.4f}",
            val_loss=f"{val_loss:.4f}",
            auc=f"{auc:.4f}",
            f1=f"{f1:.4f}",
            patience=f"{counter}/{max_val_decrease}",
        )

        print(
            f"Epoch: {epoch:03d}, Loss: {train_loss:.4f}, "
            f"[Avg,Max,Min] batch size: [{np.mean(all_batch_size):.4f},{np.max(all_batch_size):.4f},{np.min(all_batch_size):.4f}]"
        )

    setting = {"lr": lr}
    setting.update(hyperparameters)
    print("Model trained")

    if save_model_bool:
        model = torch.load(output_dir + f"/models_temp/{name_experiment}/model.pt")

    auc, precision, recall, acc, f1, val_loss = evaluate(model, test_loader, predict_edge=predict_edge)
    precision_at_10, recall_at_10, average_precision_score_at_10, ndcg_score_at_10, mrr, recall_at_10_ts = evaluate_top_k(
        model,
        test_loader,
        all_data,
        name_experiment,
        output_dir=output_dir,
        predict_edge=predict_edge,
        num_neigh=num_neigh,
        ts_loader=use_ts_loader,
        strategy=strategy,
        error_analysis=error_analysis,
    )

    writer.add_hparams(
        setting,
        {
            "hparam/AUC": float(auc),
            "hparam/Precision": float(precision),
            "hparam/Recall": float(recall),
            "hparam/accuracy": float(acc),
            "hparam/F1": float(f1),
            "hparam/val_loss": float(val_loss),
            "precision_at_10": float(precision_at_10),
            "recall_at_10": float(recall_at_10),
            "average_precision_score_at_10": float(average_precision_score_at_10),
            "ndcg_score_at_10": float(ndcg_score_at_10),
            "MRR": float(mrr),
            "recall_at_10_ts": float(recall_at_10_ts),
        },
    )

    print("Model evaluated")
    return model


In [4]:
graph_abl_list = config["model_list_abl"][:8]
use_time_nodes = bool(config["model_list_abl"][8])

data = build_graph(
    config["data_dir"],
    abl_list=graph_abl_list,
    candidature_node=config["use_candidature_node"],
    ts_nodes=config["use_temporal"],
    ts_nodes_all=config["ts_nodes_all"],
    ts_attr=use_time_nodes,
    error_analysis=str(output_dir / "error_analysis" / config["experiment_name"])
    if config["error_analysis"]
    else None,
)
print("Graph built")




Data loaded
Nodes loaded
Edges loaded
Features loaded
Number of edges from candidature to job 53600
min_index :  0
edge_index_candidature_to_job :  tensor([[     0,      1,      2,  ...,  53597,  53598,  53599],
        [ 17817,  67270,  40131,  ..., 131289,  86298, 127635]])
edge_index_user_to_candidature :  tensor([[ 55877,  46906,  77000,  ..., 169669, 115735, 196593],
        [     0,      1,      2,  ...,  53597,  53598,  53599]])
torch.Size([2, 53600])
edge_index_user_to_skill tensor([[     0,      0,      0,  ..., 210248, 210249, 210249],
        [    27,     57,     60,  ...,     47,     51,     47]])
torch.Size([2, 962315])
Graph built


In [5]:
model = Model(
    hidden_channels=config["hidden_channels"],
    data=data,
    remove_feat=config["remove_feat"],
    freeze=config["freeze"],
    list_abl=config["model_list_abl"],
    predict_edge=config["predict_edge"],
    batch_norm=config["batch_norm"],
    linear_unit_label=config["linear_unit"],
    num_layers=config["num_layers"],
    conv_operator=config["conv_operator"],
).to(device)
print("Model instantiated")


Model instantiated


In [6]:
if config["use_ts_loader"]:
    train_loader = ts_loader(
        data,
        0,
        80,
        num_neigh=config["num_neigh"],
        predict_edge=config["predict_edge"],
        strategy=config["strategy"],
        batch_size=config["batch_size"],
    )
    test_loader = ts_loader(
        data,
        90,
        100,
        num_neigh=config["num_neigh"],
        predict_edge=config["predict_edge"],
        strategy=config["strategy"],
        batch_size=config["eval_batch_size"],
    )
    val_loader = ts_loader(
        data,
        80,
        90,
        num_neigh=config["num_neigh"],
        predict_edge=config["predict_edge"],
        strategy=config["strategy"],
        batch_size=config["eval_batch_size"],
    )
else:
    train_data, val_data, test_data = split_data(data, predict_edge=config["predict_edge"])
    train_loader = traintestval_loader(
        train_data,
        num_neigh=config["num_neigh"],
        predict_edge=config["predict_edge"],
        strat=config["strategy"],
        temporal=config["use_temporal"],
        batch_size=config["batch_size"],
        neg_sampling_ratio=1,
    )
    test_loader = traintestval_loader(
        test_data,
        num_neigh=config["num_neigh"],
        predict_edge=config["predict_edge"],
        strat=config["strategy"],
        temporal=config["use_temporal"],
        batch_size=config["eval_batch_size"],
        neg_sampling_ratio=1,
    )
    val_loader = traintestval_loader(
        val_data,
        num_neigh=config["num_neigh"],
        predict_edge=config["predict_edge"],
        strat=config["strategy"],
        temporal=config["use_temporal"],
        batch_size=config["eval_batch_size"],
        neg_sampling_ratio=1,
    )

print("Loaders ready")

edge_label_time : tensor([1704047360, 1704047360, 1704047360,  ..., 1729189760, 1729189760,
        1729189760])
edge_label_time.size() : torch.Size([107200])
edge_label_index[0].size() : torch.Size([107200])
edge_label_index[1].size() : torch.Size([107200])
edge_label_time : tensor([1704047360, 1704047360, 1704047360,  ..., 1729189760, 1729189760,
        1729189760])
edge_label_time.size() : torch.Size([107200])
edge_label_index[0].size() : torch.Size([107200])
edge_label_index[1].size() : torch.Size([107200])
edge_label_time : tensor([1704047360, 1704047360, 1704047360,  ..., 1729189760, 1729189760,
        1729189760])
edge_label_time.size() : torch.Size([107200])
edge_label_index[0].size() : torch.Size([107200])
edge_label_index[1].size() : torch.Size([107200])
Loaders ready


In [7]:
cache_dir = output_dir / 'cache' / config['experiment_name']
cache_dir.mkdir(parents=True, exist_ok=True)

torch.save(data, cache_dir / 'data.pt')
torch.save(train_loader, cache_dir / 'train_loader.pt')
torch.save(val_loader, cache_dir / 'val_loader.pt')
torch.save(test_loader, cache_dir / 'test_loader.pt')

print(f'Cached graph and loaders to {cache_dir}')


Cached graph and loaders to output\cache\hardcoded_shortlist_job_notebook


In [8]:
# cache_dir = output_dir / 'cache' / config['experiment_name']

# data = torch.load(cache_dir / 'data.pt', weights_only=False)
# train_loader = torch.load(cache_dir / 'train_loader.pt', weights_only=False)
# val_loader = torch.load(cache_dir / 'val_loader.pt', weights_only=False)
# test_loader = torch.load(cache_dir / 'test_loader.pt', weights_only=False)

# print(f'Loaded graph and loaders from {cache_dir}')


In [9]:
trained_model = train_notebook(
    model,
    train_loader,
    val_loader,
    test_loader,
    data,
    config["experiment_name"],
    strategy=config["strategy"],
    output_dir=str(output_dir),
    wd=config["wd"],
    predict_edge=config["predict_edge"],
    num_neigh=config["num_neigh"],
    use_ts_loader=config["use_ts_loader"],
    save_model_bool=config["save_model"],
    train_metric=config["train_metric"],
    max_epoch=config["max_epoch"],
    max_val_decrease=config["max_val_decrease"],
    lr=config["lr"],
    hyperparameters=config,
    error_analysis=config["error_analysis"],
)


100%|██████████| 29/29 [00:06<00:00,  4.34it/s]


[0.22622898 0.22614999 0.20356028 ... 0.22305378 0.14208242 0.31590763]
false_positive_rate:  [0.00000000e+00 0.00000000e+00 3.66300366e-04 ... 9.99816850e-01
 9.99816850e-01 1.00000000e+00]
true_positive_rate:  [0.00000000e+00 1.83150183e-04 1.83150183e-04 ... 9.99633700e-01
 1.00000000e+00 1.00000000e+00]
thresholds:  [       inf 0.44730806 0.42845267 ... 0.13193022 0.13149846 0.12804241]

Validation AUC: 0.5119
Validation Precision: 0.5000
Validation Recall: 1.0000
Validation Accuracy: 0.5000
Validation F1: 0.6667
Validation total loss: 0.7007


100%|██████████| 29/29 [00:05<00:00,  5.15it/s]


[0.15271153 0.18477383 0.12668036 ... 0.17742091 0.16979071 0.2026703 ]
false_positive_rate:  [0.00000000e+00 1.83150183e-04 1.83150183e-04 ... 8.97069597e-01
 9.16666667e-01 1.00000000e+00]
true_positive_rate:  [0.00000000e+00 0.00000000e+00 1.83150183e-04 ... 1.00000000e+00
 1.00000000e+00 1.00000000e+00]
thresholds:  [           inf 3.63712817e-01 3.62186044e-01 ... 4.41364199e-02
 1.18485627e-06 0.00000000e+00]

Validation AUC: 0.5488
Validation Precision: 0.5217
Validation Recall: 1.0000
Validation Accuracy: 0.5417
Validation F1: 0.6857
Validation total loss: 0.6924


Epochs:   3%|▎         | 1/30 [02:58<1:26:11, 178.33s/it, auc=0.5488, f1=0.6857, patience=0/20, train_loss=0.6364, val_loss=0.6924]

...

Epochs:  77%|███████▋  | 23/30 [1:12:52<21:46, 186.63s/it, auc=0.5130, f1=0.4278, patience=19/20, train_loss=0.6149, val_loss=0.7154]

[0.         0.         0.66801333 ... 0.         0.         0.        ]
false_positive_rate:  [0.00000000e+00 1.83150183e-04 9.15750916e-04 ... 3.35531136e-01
 3.35897436e-01 1.00000000e+00]
true_positive_rate:  [0.         0.         0.         ... 0.36355311 0.36355311 1.        ]
thresholds:  [           inf 9.99993503e-01 9.99961138e-01 ... 3.34354118e-05
 3.39773646e-06 0.00000000e+00]

Validation AUC: 0.5130
Validation Precision: 0.5198
Validation Recall: 0.3636
Validation Accuracy: 0.5138
Validation F1: 0.4278
Validation total loss: 0.7154
Epoch: 022, Loss: 0.6149, [Avg,Max,Min] batch size: [127.8234,128.0000,10.0000]


Epochs:  80%|████████  | 24/30 [1:16:05<19:01, 190.23s/it, auc=0.5095, f1=0.3858, patience=20/20, train_loss=0.6141, val_loss=0.7088]


[0. 0. 0. ... 0. 0. 0.]
false_positive_rate:  [0.         0.         0.         ... 0.29047619 0.29047619 1.        ]
true_positive_rate:  [0.00000000e+00 1.83150183e-04 3.66300366e-04 ... 3.08058608e-01
 3.08424908e-01 1.00000000e+00]
thresholds:  [           inf 9.99998748e-01 9.99992311e-01 ... 8.56855479e-07
 1.57363260e-07 0.00000000e+00]

Validation AUC: 0.5095
Validation Precision: 0.5150
Validation Recall: 0.3084
Validation Accuracy: 0.5090
Validation F1: 0.3858
Validation total loss: 0.7088
Epoch: 023, Loss: 0.6141, [Avg,Max,Min] batch size: [127.8234,128.0000,10.0000]
Model trained


UnpicklingError: Weights only load failed. This file can still be loaded, to do so you have two options, [1mdo those steps only if you trust the source of the checkpoint[0m. 
	(1) In PyTorch 2.6, we changed the default value of the `weights_only` argument in `torch.load` from `False` to `True`. Re-running `torch.load` with `weights_only` set to `False` will likely succeed, but it can result in arbitrary code execution. Do it only if you got the file from a trusted source.
	(2) Alternatively, to load with `weights_only=True` please check the recommended steps in the following error message.
	WeightsUnpickler error: Unsupported global: GLOBAL model.Model was not an allowed global by default. Please use `torch.serialization.add_safe_globals([model.Model])` or the `torch.serialization.safe_globals([model.Model])` context manager to allowlist this global if you trust this class/function.

Check the documentation of torch.load to learn more about types accepted by default with weights_only

In [ ]:
from pathlib import Path
import torch

save_dir = Path("output/final_models")
save_dir.mkdir(parents=True, exist_ok=True)

save_path = save_dir / f"{config['experiment_name']}_full_model.pt"
torch.save(trained_model, save_path, pickle_protocol=4)

print(f"Saved full model to: {save_path}")
